# 01. Exploratory Data Analysis (EDA) - Fashion Sales & Seasonality
### AI-Based Clothing Sales Forecasting & Inventory Optimization System

This notebook conducts an in-depth exploratory analysis of 18 months of historical fashion sales data across 30 SKUs and 5 garment categories (`Outerwear`, `Knitwear`, `Tops`, `Bottoms`, `Dresses`).

#### Objectives:
1. Inspect data integrity, missing values, and schema distributions.
2. Visualize category-level sales volumes and annual seasonality patterns.
3. Analyze weekly shopping patterns (day-of-week effect) and price elasticity during promotional discounts.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load clean processed sales
sales_path = '../data/processed/clean_sales.csv'
if not os.path.exists(sales_path):
    sales_path = 'data/processed/clean_sales.csv'

df_sales = pd.read_csv(sales_path)
df_sales['date'] = pd.to_datetime(df_sales['date'])
print(f"Loaded {len(df_sales):,} sales records across {df_sales['sku'].nunique()} unique SKUs.")
df_sales.head()

## 1. Summary Statistics & Category Aggregations

In [ ]:
cat_summary = df_sales.groupby('category').agg(
    total_units=('quantity_sold', 'sum'),
    mean_daily_sales=('quantity_sold', 'mean'),
    avg_price=('unit_price', 'mean'),
    sku_count=('sku', 'nunique')
).reset_index()
cat_summary

## 2. Category Demand Seasonality Over Time

In [ ]:
plt.figure(figsize=(14, 6))
cat_ts = df_sales.groupby(['date', 'category'])['quantity_sold'].sum().unstack()
cat_ts.rolling(7).mean().plot(linewidth=2, alpha=0.85)
plt.title('7-Day Rolling Average Daily Sales by Fashion Category', fontsize=14, fontweight='bold')
plt.ylabel('Daily Units Sold')
plt.xlabel('Date')
plt.legend(title='Category', frameon=True)
plt.tight_layout()
plt.show()

## 3. Day of Week Shopping Behavior & Promotions

In [ ]:
df_sales['day_name'] = df_sales['date'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 5))
sns.barplot(data=df_sales, x='day_name', y='quantity_sold', order=days_order, palette='viridis', errorbar=None)
plt.title('Average Units Sold by Day of Week (Weekend Demand Surge)', fontsize=13, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Mean Daily Units Sold')
plt.tight_layout()
plt.show()